# Установка моделей

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Установка токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1283.83it/s]


In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend



class Generation:
    def __init__(self,
                 llm_1: GenerationMixin, 
                 llm_2: GenerationMixin, 
                 tok_1: TokenizersBackend | SentencePieceBackend, 
                 tok_2: TokenizersBackend | SentencePieceBackend,
                 top_k: int,):
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
        self.device = 'cuda'
        self.top_k = top_k
        

    def _tokenize_one_model_complete_chat(self, 
                           tokenizer_model: TokenizersBackend | SentencePieceBackend, 
                           msg: str,):
        inputs = tokenizer_model(msg,
                                 return_tensors="pt",).to(self.device)

        return inputs

    def _tokenize_one_model_start_chat(self, 
                           tokenizer_model: TokenizersBackend | SentencePieceBackend, 
                           msg: str,):
        messages = [{"role": "user", "content": msg,}]
        inputs = tokenizer_model.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )

        return inputs.to(self.device)
    
    def _tokenize_one_model_start_chat(self, 
                           tokenizer_model: TokenizersBackend | SentencePieceBackend, 
                           msg: str,):
        messages = [{"role": "user", "content": msg,}]
        inputs = tokenizer_model.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )

        return inputs.to(self.device)

    def _tokenize_all_models(self, 
                            msg: list, 
                            tokenizers: list,
                            start: bool) -> list:
        input_ids_list =[]
        for tokenizer in tokenizers:  
            if start:
                input_ids_list.append(self._tokenize_one_model_start_chat(tokenizer_model=tokenizer, msg=msg))
            else:
                input_ids_list.append(self._tokenize_one_model_complete_chat(tokenizer_model=tokenizer, msg=msg))
        return input_ids_list
    
    def _generate(self, 
                 llm_inputs: list, 
                 llms: list[GenerationMixin]) -> list:
        models_outputs_log_probs = []
        for llm_input, llm in zip(llm_inputs, llms, strict=True):
            with torch.inference_mode():
                logits = llm(**llm_input).logits[:, -1, :]
                log_probs = torch.log_softmax(logits, dim=-1)
                models_outputs_log_probs.append(log_probs)
        
        return models_outputs_log_probs

    def _get_probs(self, log_probs: list) -> tuple[torch.tensor, torch.tensor]:
        log_probs_arr = []
        token_probs_arr = []

        for lop_prob in log_probs:
            top_log_probs, top_token_ids = torch.topk(lop_prob[0],
                                                      k=self.top_k,)
            log_probs_arr.append(top_log_probs)
            token_probs_arr.append(top_token_ids)

        return log_probs_arr, token_probs_arr

    def _decode_generated_ids(self, 
                             output_ids: list, 
                             tokenizers: list[TokenizersBackend]) -> list:
        outputs_txt = []
        for output_id, tokenizer in zip(output_ids, tokenizers, strict=True):
            generated_txt = tokenizer.decode(output_id[0], skip_special_tokens=True)
            outputs_txt.append(generated_txt)

        return outputs_txt
    
    def generate_pipe(self, input_str: str, start_chat: bool, model_to_freeze: str):
        
        if model_to_freeze == '1':
            tokenizers_list = [self.tok_2]
            llms_list = [self.llm_2]
        elif model_to_freeze == '2':
            tokenizers_list = [self.tok_1]
            llms_list = [self.llm_1]
        else:
            tokenizers_list = [self.tok_1, self.tok_2]
            llms_list = [self.llm_1, self.llm_2]

        
        llm_inputs = self._tokenize_all_models(msg=input_str,
                                              tokenizers=tokenizers_list,
                                              start=start_chat)
        
        models_outputs_log_probs = self._generate(llm_inputs=llm_inputs, llms=llms_list)
        top_log_probs, top_token_ids = self._get_probs(log_probs=models_outputs_log_probs)
        models_probs_list = {}

        for model_number, (model_token_ids, model_log_probs, tokenizer,) in enumerate(
        zip(top_token_ids, top_log_probs, tokenizers_list, strict=True,),
        start=1,):
            print(f"Модель {model_number}")
            model_prob_dict = {}
            for rank, (token_id, log_prob) in enumerate(
                zip(model_token_ids, model_log_probs, strict=True),
                start=1,
            ):
                token_id = token_id.item()

                print(
                    f"{rank}. "
                    f"token={tokenizer.decode([token_id])!r}, "
                    f"id={token_id}, "
                    f"prob={log_prob.exp().item():.6f}"
                )
                model_prob_dict[tokenizer.decode([token_id])] = log_prob.exp().item()
            
            models_probs_list[f'{model_number}'] = model_prob_dict
        # print(f'{models_probs_list['1']=}, {models_probs_list['2']=}')
        if '1' in models_probs_list and '2' in models_probs_list:
             return models_probs_list['1'], models_probs_list['2']
        if '1' in models_probs_list:
            return models_probs_list['1']
        if '2' in models_probs_list:
            return models_probs_list['2']
        # return models_probs_list['1'] if '1' in models_probs_list else, models_probs_list['2']


In [4]:
import math

class Agreement:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.step_matrix = {}
        self.model_to_freeze = ''
        self.probs_generator = probs_generator
        self.input_str = input_str

    def runpipe(self):
        p = 0
        for p in range(10):
            self.step_matrix = {}
            self.prob_distribution_1, self.prob_distribution_2 = self.probs_generator.generate_pipe(input_str=self.input_str, 
                                                                                                    start_chat=True if p==0 else False,
                                                                                                    model_to_freeze=self.model_to_freeze)
            print(f'{self.prob_distribution_1=}')
            print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            self.match_tokens()
            max_key = max(self.step_matrix, key=self.step_matrix.get)
            self.input_str = self.input_str + max_key
            # freeze = self.model_to_freeze
            print(f'{self.step_matrix=}')
            print(f'очищаем step matrix')
            # print(f'{freeze=}')
        
    def sum_probs_common_token(self, token: str) -> float:
        return (math.log(self.prob_distribution_1[token]) + math.log(self.prob_distribution_2[token]))

    def exact_match_case(self, token) -> bool:
        if token in self.prob_distribution_2:
                self.step_matrix[token] = self.sum_probs_common_token(token=token)
                print(f'Нашли точное совпадение {token=} присутствует в обоих словарях')
                print(f'Матрица после суммирования вероятностей {self.step_matrix=}')
                return True

    def composite_match_case(self, token, opposite_distrib: dict[str, float], which_model_freeze: str) -> None:
        candidates = {}
        for elem in opposite_distrib:
            if elem.startswith(token) and token != elem:
                # self.model_to_freeze = which_model_freeze
                print(f'Элемент-кандидат: {token=}, во что он может выродится: {elem=}')
                if token not in candidates:                    
                    candidates[token] = [elem]
                else:
                    candidates[token].append(elem)

        print(f'Кандидатский словарь: {candidates=}')
        
        for key, value in candidates.items():
            print(f'Выполняем прогноз для {key=}')
            print(f'Строка-продолжение: {self.input_str+key}')
            tmp_distrib= self.probs_generator.generate_pipe(input_str=self.input_str+key, 
                                                                              start_chat=False,
                                                                              model_to_freeze=which_model_freeze)
            
            print(f'Генерация-наследник 1: {tmp_distrib=}')
            for key2 in tmp_distrib:
                # print(f'рассматриваем {key2=}')
                for val in value:
                    print(f'сравниваем {val=} с {key+key2=}')
                    comparison = key+key2
                    if comparison.startswith(val) and comparison != val:
                        print(f'Нашли префикс. {comparison=} начинается с {val=}')
                        self.step_matrix[val] += math.log(tmp_distrib[key2])
                        print(f'{self.step_matrix=}')
                    elif comparison == val:
                        print(f'Нашли точное совпадение в наследнике. Цикл поиска завершен. {comparison=} и {val=}')
                        if val not in self.step_matrix:
                            if key in self.prob_distribution_1:
                                self.step_matrix[val] = math.log(self.prob_distribution_1[key])+math.log(tmp_distrib[key2])
                            elif key in self.prob_distribution_2:
                                self.step_matrix[val] = math.log(self.prob_distribution_2[key])+math.log(tmp_distrib[key2])
                        print(f'{self.step_matrix}')
            print(f'################################')
                
    def match_tokens(self) -> None:
        for token_1 in self.prob_distribution_1:
            exact_match = self.exact_match_case(token_1)
            if not exact_match:
                self.composite_match_case(token_1, opposite_distrib=self.prob_distribution_2, which_model_freeze='1')
        for token_2 in self.prob_distribution_2:
            self.composite_match_case(token_2, opposite_distrib=self.prob_distribution_1, which_model_freeze='2')

In [ ]:
class Agreement2:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.step_matrix = {}
        self.model_to_freeze = ''
        self.probs_generator = probs_generator
        self.input_str = input_str
    
    def runpipe(self):
        p = 0
        for p in range(10):
            self.step_matrix = {}
            print('###############################################################')
            print(f'Подаем на вход:')
            print(f'{self.input_str}')
            print('###############################################################')
            self.prob_distribution_1, self.prob_distribution_2 = self.probs_generator.generate_pipe(input_str=self.input_str, 
                                                                                                    start_chat=True if p==0 else False,
                                                                                                    model_to_freeze=self.model_to_freeze)
            print(f'{self.prob_distribution_1=}')
            print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            max_token = self.match_chars()
            self.input_str = self.input_str + max_token

    def count_min_token_len_per_distrib(self, distribs: list):
        distrib_min_len = {}
        for idx, distrib in enumerate(distribs):
            distrib_min_len[idx] = min(set([len(key) for key in distrib]))
        return distrib_min_len

    def find_the_suitest_token(self, distrib: dict, char_num: int):
        char_prob = {}
        for token, probability in distrib.items():
            if token[char_num] not in char_prob:
                char_prob[token[char_num]] = probability
            else:
                char_prob[token[char_num]] += probability
        
        if len(char_prob) == len(distrib):
            return char_prob
        
        the_most_common_char = max(char_prob, key=char_prob.get)
        return the_most_common_char

    def ensemble(self, ensemble_distr: dict):
        token_prob = {}        
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        print(f'{token_prob=}')
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token


    def match_chars(self):
        probs_list = [self.prob_distribution_1, self.prob_distribution_2]
        distrib_min_len = self.count_min_token_len_per_distrib(distribs=probs_list)
        print(f'{distrib_min_len=}')

        ensemble_dict = {}

        for idx, (distrib, min_len) in enumerate(zip(probs_list, distrib_min_len.values())):
            for char_num in range(min_len):
                the_most_common_char = self.find_the_suitest_token(distrib=distrib, char_num=char_num)
                print(f'{the_most_common_char=}')
                if isinstance(the_most_common_char, dict):
                    distrib = the_most_common_char
                    ensemble_dict[idx] = distrib

        the_most_popular_token = self.ensemble(ensemble_distr=ensemble_dict)
        return the_most_popular_token
            

In [120]:
agg = Generation(llm_1=model_1, llm_2=model_2, tok_1=tokenizer_1, tok_2=tokenizer_2, top_k=10)
models_probs_list = agg.generate_pipe("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Natal", start_chat=False, model_to_freeze='')

Модель 1
1. token='ia', id=972, prob=0.995605
2. token='ias', id=3832, prob=0.002430
3. token='ie', id=882, prob=0.000789
4. token='ian', id=1184, prob=0.000175
5. token='ja', id=3039, prob=0.000092
6. token='ya', id=5142, prob=0.000077
7. token='iae', id=28708, prob=0.000070
8. token='iam', id=3598, prob=0.000061
9. token='ys', id=1264, prob=0.000036
10. token='ians', id=2858, prob=0.000033
Модель 2
1. token='ia', id=685, prob=1.000000
2. token='IA', id=5863, prob=0.000006
3. token='ina', id=2210, prob=0.000003
4. token='ya', id=7755, prob=0.000002
5. token='ias', id=3473, prob=0.000001
6. token='a', id=64, prob=0.000001
7. token='ania', id=9166, prob=0.000001
8. token='iaz', id=26975, prob=0.000001
9. token='ita', id=6255, prob=0.000001
10. token='ía', id=7431, prob=0.000000


In [121]:
agreement = Agreement2(probs_generator=agg,
                      input_str='Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?')
agreement.runpipe()

###############################################################
Подаем на вход:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
###############################################################
Модель 1
1. token='N', id=555, prob=0.975586
2. token='In', id=1286, prob=0.020554
3. token='She', id=8052, prob=0.000864
4. token='Let', id=8232, prob=0.000524
5. token=' Natal', id=34750, prob=0.000349
6. token='On', id=3312, prob=0.000244
7. token='There', id=3776, prob=0.000225
8. token='To', id=3097, prob=0.000225
9. token='First', id=9578, prob=0.000208
10. token='Nat', id=26532, prob=0.000190
Модель 2
1. token='To', id=1249, prob=0.844727
2. token='N', id=45, prob=0.090515
3. token='In', id=641, prob=0.031769
4. token='Let', id=10061, prob=0.023239
5. token='First', id=5338, prob=0.001418
6. token='1', id=16, prob=0.001005
7. token='If', id=2679, prob=0.000887
8. token='Certainly', id

ValueError: max() iterable argument is empty

In [116]:
agreement = Agreement(probs_generator=agg,
                      input_str='Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?')
agreement.runpipe()

Модель 1
1. token='N', id=555, prob=0.975586
2. token='In', id=1286, prob=0.020554
3. token='She', id=8052, prob=0.000864
4. token='Let', id=8232, prob=0.000524
5. token=' Natal', id=34750, prob=0.000349
Модель 2
1. token='To', id=1249, prob=0.844727
2. token='N', id=45, prob=0.090515
3. token='In', id=641, prob=0.031769
4. token='Let', id=10061, prob=0.023239
5. token='First', id=5338, prob=0.001418
self.prob_distribution_1={'N': 0.9755859375, 'In': 0.0205535888671875, 'She': 0.0008635520935058594, 'Let': 0.0005235671997070312, ' Natal': 0.00034880638122558594}
self.prob_distribution_2={'To': 0.8447265625, 'N': 0.09051513671875, 'In': 0.031768798828125, 'Let': 0.0232391357421875, 'First': 0.0014181137084960938}
Нашли точное совпадение token='N' присутствует в обоих словарях
Матрица после суммирования вероятностей self.step_matrix={'N': -2.426955212652202}
Нашли точное совпадение token='In' присутствует в обоих словарях
Матрица после суммирования вероятностей self.step_matrix={'N': -2.

ValueError: max() iterable argument is empty

In [23]:
class Pipeline:
    def __init__(self, generator: Generation):
        self.generator = generator

    def runpipe(self, input_str: str):
        freeze = ''
        for p in range(1000):
            models_probs_list = self.generator.generate_pipe(input_str=input_str, start_chat=True if p==0 else False,
                                                             model_to_freeze=freeze)
            
            agreement = Agreement(prob_distribution_1=models_probs_list['1'],
                                  prob_distribution_2=models_probs_list['2'])

            agreement.match_tokens()
            max_key = max(agreement.step_matrix, key=agreement.step_matrix.get)
            input_str = input_str + max_key
            freeze = agreement.model_to_freeze
            print(f'{freeze=}')

In [24]:
pipe = Pipeline(generator=agg)
pipe.runpipe(input_str="Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?")

Модель 1
1. token='N', id=555, prob=0.975586
2. token='In', id=1286, prob=0.020554
3. token='She', id=8052, prob=0.000864
4. token='Let', id=8232, prob=0.000524
5. token=' Natal', id=34750, prob=0.000349
Модель 2
1. token='To', id=1249, prob=0.844727
2. token='N', id=45, prob=0.090515
3. token='In', id=641, prob=0.031769
4. token='Let', id=10061, prob=0.023239
5. token='First', id=5338, prob=0.001418
----------------------------


TypeError: Agreement.__init__() got an unexpected keyword argument 'prob_distribution_1'

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class GenerationLoop:
    def __init__(self, 
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizersBackend | SentencePieceBackend,
                 tokenizer_2: TokenizersBackend | SentencePieceBackend,
                 mapping_vocab: dict):

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class Benchmark:
    def __init__(self,
                 map_proj: dict, 
                 map_mlp: dict, 
                 llm_1: GenerationMixin, 
                 llm_2: GenerationMixin, 
                 tok_1: TokenizersBackend | SentencePieceBackend, 
                 tok_2: TokenizersBackend | SentencePieceBackend,):
        self.map_proj = map_proj
        self.map_mlp = map_mlp
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
    
    def projecting_loop(self, tokens):
        projected_idx_mlp = []
        projected_idx_proj = []
        for idx in tokens:
            token = self.tok_1.decode(idx)
            idx = int(idx)
            print(f'id "{idx}", строка "{token}"')
            if token == '\n': 
                token = 'Ċ'
            if token.startswith(' '):
                token = token.replace(' ',"Ġ")
            
            mlp_projection = self.map_mlp[(token, idx)]
            mtrx_projection = self.map_proj[(token, idx)]

            print(f'НС-отображение {mlp_projection}') 
            print(f'Проекционное отображение {mtrx_projection}') 
            
            projected_idx_mlp.append(mlp_projection[0][1])

            if len(mtrx_projection) > 1:
                for elem in mtrx_projection:
                    print(f'{elem=}')
                    projected_idx_proj.append(elem[1])
            else:
                projected_idx_proj.append(mtrx_projection[0][1])
            print('##########################################')
        

        # print(f'{projected_arr_mlp=}')
        # print(f'{projected_arr_proj=}')
        projected_only_idx_mlp = []
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_mlp)}')
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_proj)}')


    def generation_pipe(self, msg: list):
        text = self.tok_1.apply_chat_template(msg,
                                              tokenize=True,
                                              return_tensors='pt',
                                              add_generation_prompt=False)
        
        tokens_1 = text['input_ids'][0]
        strings = self.tok_1.decode(tokens_1, skip_special_tokens=True)
        return tokens_1, strings

    def map_input(self, input: str):
        message = [{'role': 'user', 'content': input}]

        tokens, strings = self.generation_pipe(msg=message)

        print(f'Разбивка по токенам: {tokens=}')
        print(f'Декодировано в текст: {strings=}')

        self.projecting_loop(tokens)

In [ ]:
bench = Benchmark(map_proj=proj_mapping,
                  map_mlp=mlp_mapping,
                  llm_1=model_1,
                  llm_2=model_2,
                  tok_1=tokenizer_model_1,
                  tok_2=tokenizer_model_2)

bench.map_input(input="The file, 90 megabytes in size, downloads at the rate of 5 megabytes per second for its first 60 megabytes, and then 10 megabytes per second thereafter. How long, in seconds, does it take to download entirely?")